In [ ]:
!pip install streamlit pyngrok pandas numpy

In [ ]:
%%writefile analytics_engine.py

import pandas as pd
import numpy as np

def find_col(df, possible_names):
    for name in possible_names:
        for col in df.columns:
            if name.lower() == col.lower():
                return col
    for name in possible_names:
        for col in df.columns:
            if name.lower() in col.lower():
                return col
    return None


# COLUMN FIX
# -----------------------------
def get_profit_col(df):
    if 'Order Profit Per Order' in df.columns:
        return 'Order Profit Per Order'
    elif 'Benefit per order' in df.columns:
        return 'Benefit per order'
    return find_col(df, ['profit'])


def get_discount_rate_col(df):
    if 'Order Item Discount Rate' in df.columns:
        return 'Order Item Discount Rate'
    return find_col(df, ['discount rate'])



# LOAD DATA
# -----------------------------
def load_data(file):
    try:
        df = pd.read_csv(file, encoding='utf-8')
    except:
        df = pd.read_csv(file, encoding='latin1')

    df.columns = (
        df.columns.str.strip()
        .str.replace('\xa0',' ',regex=True)
        .str.replace('\n',' ',regex=True)
        .str.replace('_',' ')
    )

    for col in df.columns:
        if any(x in col.lower() for x in ['sales','profit','discount']):
            df[col] = pd.to_numeric(df[col], errors='coerce')

    sales = find_col(df, ['Sales'])
    profit = get_profit_col(df)

    if sales and profit:
        df = df.dropna(subset=[sales, profit])
        df = df[df[sales] > 0]

    return df



# KPI
# -----------------------------
def compute_kpis(df):
    sales = find_col(df, ['Sales'])
    profit = get_profit_col(df)
    discount = find_col(df, ['Order Item Discount'])

    if sales is None or profit is None:
        return {"Revenue":0,"Profit":0,"Margin":0,"Discount Impact Ratio":0}

    revenue = df[sales].sum()
    prof = df[profit].sum()

    margin = (prof/revenue)*100 if revenue else 0

    discount_ratio = (
        df[discount].sum()/revenue*100
        if discount and revenue else 0
    )

    return {
        "Revenue": revenue,
        "Profit": prof,
        "Margin": margin,
        "Discount Impact Ratio": discount_ratio
    }



# TREND
# -----------------------------
def trend_analysis(df):
    date = find_col(df, ['Order Date','Date'])
    sales = find_col(df, ['Sales'])
    profit = get_profit_col(df)

    if None in [date,sales,profit]:
        return None

    df[date] = pd.to_datetime(df[date], errors='coerce')

    t = df.groupby(df[date].dt.to_period('M')).agg({
        sales:'sum', profit:'sum'
    }).reset_index()

    t[date] = t[date].astype(str)
    return t


# CUSTOMER
# -----------------------------
def customer_analysis(df):
    cust = find_col(df, ['Customer Id'])
    sales = find_col(df, ['Sales'])
    profit = get_profit_col(df)

    if None in [cust,sales,profit]:
        return pd.DataFrame()

    c = df.groupby(cust).agg({sales:'sum', profit:'sum'}).reset_index()

    c['Customer Value Index'] = c[profit]/c[sales]

    c['Score'] = (
        (c[sales]/c[sales].max()) +
        (c[profit]/c[profit].max())
    )

    c['Segment'] = pd.qcut(
        c['Score'], q=3,
        labels=['Low','Medium','High'],
        duplicates='drop'
    )

    return c.sort_values(by=profit, ascending=False)


def profit_concentration(cust):
    if cust.empty:
        return 0
    return (cust.iloc[:10,2].sum()/cust.iloc[:,2].sum())*100



# PRODUCT
# -----------------------------
def product_analysis(df):
    prod = find_col(df,['Product Name'])
    sales = find_col(df,['Sales'])
    profit = get_profit_col(df)

    if None in [prod,sales,profit]:
        return pd.DataFrame(),pd.DataFrame(),pd.DataFrame()

    p = df.groupby(prod).agg({sales:'sum',profit:'sum'}).reset_index()

    p['Margin (%)'] = (p[profit]/p[sales])*100

    loss = p[p['Margin (%)']<0]

    risky = p[
        (p[sales]>p[sales].quantile(0.75)) &
        (p['Margin (%)']<p['Margin (%)'].quantile(0.25))
    ]

    return p.sort_values(by='Margin (%)',ascending=False), loss, risky



# CATEGORY
# -----------------------------
def category_analysis(df):
    cat = find_col(df,['Category Name'])
    sales = find_col(df,['Sales'])
    profit = get_profit_col(df)
    market = find_col(df,['Market'])

    if None in [cat,sales,profit]:
        return pd.DataFrame(), None

    c = df.groupby(cat).agg({sales:'sum',profit:'sum'}).reset_index()
    c['Margin (%)'] = (c[profit]/c[sales])*100

    heat = None
    if market:
        temp = df.copy()
        temp['Margin'] = temp[profit]/temp[sales]
        heat = temp.pivot_table(values='Margin', index=cat, columns=market)

    return c, heat



# DISCOUNT
# -----------------------------
def discount_analysis(df, slider):
    df = df.copy()

    sales = find_col(df,['Sales'])
    profit = get_profit_col(df)
    disc = get_discount_rate_col(df)

    if None in [sales,profit]:
        return df

    df['Profit Ratio'] = df[profit]/df[sales]
    df['Simulated Profit'] = df[profit]*(1-slider)

    if disc:
        df['Discount Rate'] = df[disc]

    return df



# MARKET
# -----------------------------
def market_analysis(df):
    market = find_col(df,['Market'])
    sales = find_col(df,['Sales'])
    profit = get_profit_col(df)

    if None in [market,sales,profit]:
        return pd.DataFrame()

    m = df.groupby(market).agg({sales:'sum',profit:'sum'}).reset_index()
    m['Margin (%)'] = (m[profit]/m[sales])*100
    return m

# REGION
def region_analysis(df):
    region = find_col(df,['Order Region'])
    sales = find_col(df,['Sales'])
    profit = get_profit_col(df)

    if None in [region,sales,profit]:
        return pd.DataFrame()

    r = df.groupby(region).agg({sales:'sum',profit:'sum'}).reset_index()
    r['Margin (%)'] = (r[profit]/r[sales])*100
    return r

# COUNTRY
def country_analysis(df):
    country = find_col(df,['Order Country'])
    sales = find_col(df,['Sales'])
    profit = get_profit_col(df)

    if None in [country,sales,profit]:
        return pd.DataFrame()

    c = df.groupby(country).agg({sales:'sum',profit:'sum'}).reset_index()
    c['Margin (%)'] = (c[profit]/c[sales])*100
    return c



# SEGMENT
# -----------------------------
def segment_contribution(df):
    seg = find_col(df,['Customer Segment'])
    profit = get_profit_col(df)

    if None in [seg,profit]:
        return pd.Series()

    return df.groupby(seg)[profit].sum()

Overwriting analytics_engine.py
